# 🔍 Optional: Token Inspection — Debug Notebook

Use this notebook to manually obtain and inspect tokens at each step of the authentication chain.

This helps diagnose issues like:
- Missing `groupIds` claim in the exchanged token
- Wrong audience in the token sent to Vault
- Group membership not being resolved by Vault

---

## Steps

1. **Get the user's access token** from the frontend (copy from browser)
2. **Get the agent token** via client credentials
3. **Perform token exchange** and inspect the result
4. **Authenticate with Vault** and inspect the resulting Vault token policies
5. **Attempt credential request** for `hr-basic-reader` and `hr-admin-reader`

---

## ⚙️ Step 0: Configuration

Fill in the values below. All credentials are loaded from your `.env` file.

In [ ]:
import os
import sys
import json
import base64
import requests
from pathlib import Path

# Load .env from project root
lab_root = Path.cwd().parent
sys.path.insert(0, str(lab_root))
os.chdir(lab_root)

from dotenv import load_dotenv
load_dotenv()

# IBM Verify config
TOKEN_ENDPOINT = os.getenv('IBM_VERIFY_OIDC_URL', '').rstrip('/') + '/token'

AGENT_CLIENT_ID     = os.getenv('IBM_VERIFY_AGENT_CLIENT_ID', '')
AGENT_CLIENT_SECRET = os.getenv('IBM_VERIFY_AGENT_CLIENT_SECRET', '')

EXCHANGE_CLIENT_ID     = os.getenv('IBM_VERIFY_TOOL_CLIENT_ID', '')
EXCHANGE_CLIENT_SECRET = os.getenv('IBM_VERIFY_TOOL_CLIENT_SECRET', '')

# Vault config
VAULT_ADDR = os.getenv('VAULT_ADDR', 'http://localhost:8200')

print(f"Token endpoint : {TOKEN_ENDPOINT}")
print(f"Agent client   : {AGENT_CLIENT_ID[:20]}..." if AGENT_CLIENT_ID else "Agent client   : NOT SET")
print(f"Exchange client: {EXCHANGE_CLIENT_ID[:20]}..." if EXCHANGE_CLIENT_ID else "Exchange client: NOT SET")
print(f"Vault address  : {VAULT_ADDR}")

## 🛠️ Helper: Decode and Pretty-Print a JWT

This function decodes a JWT without verifying the signature, so you can inspect its claims.

In [5]:
def decode_jwt(token: str, label: str = "Token") -> dict:
    """Decode a JWT and print its claims. No signature verification."""
    try:
        parts = token.split('.')
        if len(parts) != 3:
            print(f"❌ {label}: Not a valid JWT (expected 3 parts, got {len(parts)})")
            return {}
        
        # Decode payload (add padding if needed)
        payload_b64 = parts[1]
        padding = 4 - len(payload_b64) % 4
        if padding != 4:
            payload_b64 += '=' * padding
        
        claims = json.loads(base64.urlsafe_b64decode(payload_b64))
        
        print(f"\n{'='*60}")
        print(f"📋 {label} Claims:")
        print('='*60)
        print(json.dumps(claims, indent=2))
        print('='*60)
        
        # Highlight key fields
        print(f"\n🔑 Key fields:")
        print(f"  sub      : {claims.get('sub', '❌ MISSING')}")
        print(f"  aud      : {claims.get('aud', '❌ MISSING')}")
        print(f"  groupIds : {claims.get('groupIds', '❌ MISSING — groups will NOT be resolved by Vault!')}")
        print(f"  groups   : {claims.get('groups', '(not present)')}")
        print(f"  act      : {claims.get('act', '(not present)')}")
        print(f"  email    : {claims.get('email', '(not present)')}")
        
        return claims
    except Exception as e:
        print(f"❌ Failed to decode {label}: {e}")
        return {}

print("✅ Helper function loaded")

✅ Helper function loaded


---
## 🪪 Step 1: Inspect the User's Tokens

The callback response contains two tokens — inspect both:

- **`access_token`** — the user's access token (JWT format). This is what the Vault Credentials Tool uses as the **subject token** for the token exchange.
- **`id_token`** — the user's ID token. Contains identity claims but may have different claims than the access token.

**How to get the tokens:**

1. Open **Browser DevTools** (F12 or Cmd+Option+I) and go to the **Network** tab — do this **before** logging in
2. Open the AskHR frontend at http://localhost:3000
3. Log in as `bob.anderson` (or `alice.smith`)
4. After the login redirect completes, in the Network tab filter by **`callback`**
5. Find the `POST` request to `http://localhost:5001/api/auth/callback`
6. Click on it → go to the **Response** tab
7. Copy both the **`access_token`** and **`id_token`** values and paste them below

> 💡 The tokens are returned directly in the backend callback response JSON. They are stored server-side in the Flask session — they are **not** in localStorage or cookies visible in the browser.

In [6]:
# Paste the access_token from the callback response here
USER_ACCESS_TOKEN = "eyJhbGciOiJSUzI1NiIsImtpZCI6InNlcnZlciIsInR5cCI6ImF0K2p3dCJ9.eyJhcHBfaWQiOiI4MzgyNjMxNzkwNTEwMzAyODk1IiwiYXVkIjpbIjcyYmE2NWE2LWY1ZDAtNDVmMy1iMTljLTIxZmJkZmU3NjQ2YSJdLCJjbGllbnRfaWQiOiI3MmJhNjVhNi1mNWQwLTQ1ZjMtYjE5Yy0yMWZiZGZlNzY0NmEiLCJleHAiOjE3NzI0NDM2ODMsImdyYW50X2lkIjoiMzgyY2ZiMDItNWM2OS00NzJmLWE1MDgtYzgzYjg3MjhlYzNmIiwiZ3JhbnRfdHlwZSI6ImF1dGhvcml6YXRpb25fY29kZSIsImdyb3VwSWRzIjpbImhyLWFkbWluIiwiYWxsVXNlcnMiXSwiaWF0IjoxNzcyNDQwMDgzLCJpc3MiOiJodHRwczovL2F1dG9tYXRpb24udmVyaWZ5LmlibS5jb20vb2lkYy9lbmRwb2ludC9kZWZhdWx0IiwianRpIjoiMzg4YjM4NWItYmE5MS00MDU0LThmMTgtZGJhY2U2MDQ3N2MyLk0xOHhOemN5TkRRd01EZ3pYekU0IiwibWF5X2FjdCI6eyJjbGllbnRfaWQiOiJjYWY3N2ZhMy01ODczLTQwM2UtOTAwZS1kOGIwYzJhOGU1NDAifSwibmJmIjoxNzcyNDQwMDgzLCJwcmVmZXJyZWRfdXNlcm5hbWUiOiJib2IuYW5kZXJzb24iLCJyZWFsbU5hbWUiOiJjbG91ZElkZW50aXR5UmVhbG0iLCJzY29wZSI6Im9wZW5pZCBwcm9maWxlIGVtYWlsIGdyb3VwcyIsInN1YiI6IjY0NjAwOEZXWFEiLCJ1bmlxdWVTZWN1cml0eU5hbWUiOiI2NDYwMDhGV1hRIiwidXNlclR5cGUiOiJyZWd1bGFyIn0.FrvoXxExLMdL5LLK-ani5qJy4m7pNahh0-sPlm969qfSCSQesLxuIf48kre_da4q0FD3ZFQVwVbjnzfQiM3i0NEpNWE5OIxLf2wp98CXLN887ryJc01p5c5fkjgdH1vFWKClol75Uzv2kOxaaf8dPgF_8_bJtiMKXirr6FhEwNLbyGXxG9ZFEGLruiEbEXBMA-7H1QYPUsxig6XJZN_7Dbx-UTqF8wztR5rJwfOBLF_dFCGu0p1_2fGfl8nTCwKmCmU-S1-FxU3uBxU7NS703ei5rwFeBk6J2qVDowGu5d2zQP5VQ0ts5SVDgYeLnfcaax2e270M6lXpexKQEUWovQ"

# Paste the id_token from the callback response here
USER_ID_TOKEN = "eyJhbGciOiJSUzI1NiIsImtpZCI6InNlcnZlciJ9.eyJhY3IiOiJ1cm46aWJtOnNlY3VyaXR5OnBvbGljeTppZDoxIiwiYW1yIjpbInBhc3N3b3JkIl0sImF0X2hhc2giOiJYV044X2ZVQ3NISk1BRUFUanc5cXJRIiwiYXVkIjpbIjcyYmE2NWE2LWY1ZDAtNDVmMy1iMTljLTIxZmJkZmU3NjQ2YSJdLCJhdXRoX3RpbWUiOjE3NzI0Mzk5OTksImRpc3BsYXlOYW1lIjoiYm9iIGFuZGVyc29uIiwiZW1haWwiOiJib2IuYW5kZXJzb244NTQxQGdtYWlsLmNvbSIsImVtYWlsX3ZlcmlmaWVkIjp0cnVlLCJleHAiOjE3NzI0NDcyODMsImZhbWlseV9uYW1lIjoiYW5kZXJzb24iLCJnaXZlbl9uYW1lIjoiYm9iIiwiZ3JvdXBJZHMiOlsiaHItYWRtaW4iLCJhbGxVc2VycyJdLCJpYXQiOjE3NzI0NDAwODMsImlzcyI6Imh0dHBzOi8vYXV0b21hdGlvbi52ZXJpZnkuaWJtLmNvbS9vaWRjL2VuZHBvaW50L2RlZmF1bHQiLCJqdGkiOiJhM2I5ZGNkYy0xMjMzLTRmNTgtYTA3YS03NjA5NWY0YzQ1OWUiLCJuYW1lIjoiYm9iIGFuZGVyc29uIiwicHJlZmVycmVkX3VzZXJuYW1lIjoiYm9iLmFuZGVyc29uIiwicmF0IjoxNzcyNDQwMDgyLCJyZWFsbU5hbWUiOiJjbG91ZElkZW50aXR5UmVhbG0iLCJydF9oYXNoIjoiMHQ0QkNJSUFvZ3VELV80SjRpb2ppdyIsInNfaGFzaCI6InhvSVNMMkFpbVJHQmdMa1NwRlFQU3ciLCJzdWIiOiI2NDYwMDhGV1hRIiwidW5pcXVlU2VjdXJpdHlOYW1lIjoiNjQ2MDA4RldYUSIsInVzZXJUeXBlIjoicmVndWxhciJ9.AkrUrE4IIaZCuP20InsouiQ2Aev4AN59eZCWLhlHeeklbXpHU0I9Me75gwSF-V_BQ3nC9tXB0eg8rNyvBlLmQmLmvxIsOoRigk9LQcPSLX5bPZkxmohs71AlFeXnslYtaRfc7WOeWCSKbJFKd9gZ7C7crj3RoeYCVEdWP3bEEETS9pHeJK8ESftllY2v8bxRrK7fh155Uf1Vtt6yf57ECG8VPnJSLDq7quDAEDG1wjOLWKTHLXE-bclENnRfWegqmbHNPfr8sTuE7vKSf6NKZ2IBnN81d4ter7frHGpDCii5XrBkT9ZsE7LUPORhel1k5-jSuiPCVJTbbd8jy7rQtA"

print("\n" + "="*60)
print("Inspecting access_token (used as subject token for exchange)")
print("="*60)
user_claims = decode_jwt(USER_ACCESS_TOKEN, "User Access Token")

print("\n" + "="*60)
print("Inspecting id_token")
print("="*60)
id_claims = decode_jwt(USER_ID_TOKEN, "User ID Token")

# Summary check
print("\n" + "="*60)
print("📊 Summary:")
print("="*60)
at_groups = user_claims.get('groupIds', None)
it_groups = id_claims.get('groupIds', None)
print(f"  access_token groupIds : {at_groups if at_groups is not None else '❌ MISSING'}")
print(f"  id_token     groupIds : {it_groups if it_groups is not None else '❌ MISSING'}")
print()
print("ℹ️  The Vault Credentials Tool uses the access_token as the subject token.")
print("   groupIds must be present in the access_token for Vault group resolution to work.")


Inspecting access_token (used as subject token for exchange)

📋 User Access Token Claims:
{
  "app_id": "8382631790510302895",
  "aud": [
    "72ba65a6-f5d0-45f3-b19c-21fbdfe7646a"
  ],
  "client_id": "72ba65a6-f5d0-45f3-b19c-21fbdfe7646a",
  "exp": 1772443683,
  "grant_id": "382cfb02-5c69-472f-a508-c83b8728ec3f",
  "grant_type": "authorization_code",
  "groupIds": [
    "hr-admin",
    "allUsers"
  ],
  "iat": 1772440083,
  "iss": "https://automation.verify.ibm.com/oidc/endpoint/default",
  "jti": "388b385b-ba91-4054-8f18-dbace60477c2.M18xNzcyNDQwMDgzXzE4",
  "may_act": {
    "client_id": "caf77fa3-5873-403e-900e-d8b0c2a8e540"
  },
  "nbf": 1772440083,
  "preferred_username": "bob.anderson",
  "realmName": "cloudIdentityRealm",
  "scope": "openid profile email groups",
  "sub": "646008FWXQ",
  "uniqueSecurityName": "646008FWXQ",
  "userType": "regular"
}

🔑 Key fields:
  sub      : 646008FWXQ
  aud      : ['72ba65a6-f5d0-45f3-b19c-21fbdfe7646a']
  groupIds : ['hr-admin', 'allUsers']


---
## 🤖 Step 2: Get the Agent Token (Client Credentials)

This is the actor token — the agent's own identity, obtained via client credentials.

In [7]:
print("🔐 Requesting agent token via client credentials...")

response = requests.post(
    TOKEN_ENDPOINT,
    data={
        'grant_type': 'client_credentials',
        'client_id': AGENT_CLIENT_ID,
        'client_secret': AGENT_CLIENT_SECRET,
        'scope': 'agent.act'
    },
    headers={'Content-Type': 'application/x-www-form-urlencoded'},
    timeout=30
)

print(f"HTTP Status: {response.status_code}")

if response.status_code == 200:
    AGENT_TOKEN = response.json().get('access_token')
    print("✅ Agent token obtained")
    agent_claims = decode_jwt(AGENT_TOKEN, "Agent Token (Actor)")
else:
    print(f"❌ Failed: {response.text}")
    AGENT_TOKEN = None

🔐 Requesting agent token via client credentials...
HTTP Status: 200
✅ Agent token obtained

📋 Agent Token (Actor) Claims:
{
  "app_id": "1356755185683842408",
  "aud": [
    "caf77fa3-5873-403e-900e-d8b0c2a8e540"
  ],
  "client_id": "caf77fa3-5873-403e-900e-d8b0c2a8e540",
  "exp": 1772444236,
  "grant_id": "86b87b80-a90e-4c88-95cd-d3f0421215c1",
  "grant_type": "client_credentials",
  "iat": 1772440636,
  "iss": "https://automation.verify.ibm.com/oidc/endpoint/default",
  "jti": "54ff7d4e-e9fe-4829-b886-49cbc0f46f69.M18xNzcyNDQwNjM2XzE4",
  "nbf": 1772440636,
  "realmName": "cloudIdentityRealm",
  "scope": "agent.act",
  "sub": "caf77fa3-5873-403e-900e-d8b0c2a8e540",
  "uniqueSecurityName": "caf77fa3-5873-403e-900e-d8b0c2a8e540"
}

🔑 Key fields:
  sub      : caf77fa3-5873-403e-900e-d8b0c2a8e540
  aud      : ['caf77fa3-5873-403e-900e-d8b0c2a8e540']
  groupIds : ❌ MISSING — groups will NOT be resolved by Vault!
  groups   : (not present)
  act      : (not present)
  email    : (not prese

---
## 🔄 Step 3: Perform Token Exchange

Exchange the user's access token (subject) + agent token (actor) for a delegated token scoped to Vault.

**This is the critical step** — the exchanged token must contain `groupIds` for Vault to resolve group membership.

In [17]:
print("🔄 Performing token exchange...")

exchange_data = {
    'grant_type': 'urn:ietf:params:oauth:grant-type:token-exchange',
    'subject_token': USER_ACCESS_TOKEN,
    'subject_token_type': 'urn:ietf:params:oauth:token-type:access_token',
    'requested_token_type': 'urn:ietf:params:oauth:token-type:access_token',
    'client_id': EXCHANGE_CLIENT_ID,
    'client_secret': EXCHANGE_CLIENT_SECRET,
    'audience': 'vault',
    'scope': 'openid profile email groups'
}

# Include actor token if available
if AGENT_TOKEN:
    exchange_data['actor_token'] = AGENT_TOKEN
    exchange_data['actor_token_type'] = 'urn:ietf:params:oauth:token-type:access_token'
    print("  (actor token included)")

response = requests.post(
    TOKEN_ENDPOINT,
    data=exchange_data,
    headers={'Content-Type': 'application/x-www-form-urlencoded'},
    timeout=30
)

print(f"HTTP Status: {response.status_code}")

if response.status_code == 200:
    EXCHANGED_TOKEN = response.json().get('access_token')
    print("✅ Token exchange successful")
    exchanged_claims = decode_jwt(EXCHANGED_TOKEN, "Exchanged Token (sent to Vault)")
    
    # Critical check
    if 'groupIds' in exchanged_claims:
        print(f"\n✅ groupIds preserved in exchanged token: {exchanged_claims['groupIds']}")
        print("   Vault will be able to resolve group membership.")
    else:
        print("\n❌ groupIds is MISSING from the exchanged token!")
        print("   This is why Vault cannot resolve Bob's hr-admin group.")
        print("   Fix: In IBM Verify Exchange Client → Sign-on → Endpoint configuration → Introspect")
        print("        Add attribute mapping: groupIds → groupIds")
else:
    print(f"❌ Token exchange failed: {response.text}")
    EXCHANGED_TOKEN = None

🔄 Performing token exchange...
  (actor token included)
HTTP Status: 200
✅ Token exchange successful

📋 Exchanged Token (sent to Vault) Claims:
{
  "act": {
    "client_id": "caf77fa3-5873-403e-900e-d8b0c2a8e540"
  },
  "app_id": "701279444018298144",
  "aud": [
    "a7ffd6ef-50a6-4f08-91d5-c08b37424f2f"
  ],
  "client_id": "a7ffd6ef-50a6-4f08-91d5-c08b37424f2f",
  "exp": 1772444875,
  "grant_id": "02fcbeb9-894c-48f5-8c1d-0bd3433cb1aa",
  "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
  "groupIds": [
    "hr-admin",
    "allUsers"
  ],
  "iat": 1772441274,
  "iss": "https://automation.verify.ibm.com/oidc/endpoint/default",
  "jti": "71a3b3f5-c2b8-4be2-a7ba-d0ce27b4b676.M18xNzcyNDQxMjc1XzE4",
  "nbf": 1772441274,
  "preferred_username": "bob.anderson",
  "realmName": "cloudIdentityRealm",
  "scope": "groups profile email",
  "sub": "646008FWXQ",
  "uniqueSecurityName": "646008FWXQ"
}

🔑 Key fields:
  sub      : 646008FWXQ
  aud      : ['a7ffd6ef-50a6-4f08-91d5-c08b3742

---
## 🔐 Step 4: Authenticate with Vault

Use the exchanged token to authenticate with Vault's JWT auth backend.

The response will show which **policies** Vault assigned — this tells us whether group membership was resolved.

In [18]:
if not EXCHANGED_TOKEN:
    print("❌ No exchanged token available. Complete Step 3 first.")
else:
    print("🔐 Authenticating with Vault using exchanged token...")
    
    auth_response = requests.post(
        f"{VAULT_ADDR}/v1/auth/jwt/login",
        json={
            "role": "agent-role",
            "jwt": EXCHANGED_TOKEN
        },
        timeout=10
    )
    
    print(f"HTTP Status: {auth_response.status_code}")
    
    if auth_response.status_code == 200:
        auth_data = auth_response.json()
        vault_token = auth_data['auth']['client_token']
        assigned_policies = auth_data['auth'].get('policies', [])
        token_policies = auth_data['auth'].get('token_policies', [])
        identity_policies = auth_data['auth'].get('identity_policies', [])
        metadata = auth_data['auth'].get('metadata', {})
        
        print("\n✅ Vault authentication successful")
        print(f"\n{'='*60}")
        print("📋 Vault Token Details:")
        print('='*60)
        print(f"  Token policies    : {token_policies}")
        print(f"  Identity policies : {identity_policies}")
        print(f"  All policies      : {assigned_policies}")
        print(f"  Metadata          : {metadata}")
        print('='*60)
        
        # Diagnose
        if 'hr-admin-policy' in assigned_policies:
            print("\n✅ hr-admin-policy is assigned — Bob should be able to access salary data")
        elif 'hr-basic-policy' in assigned_policies:
            print("\n⚠️  Only hr-basic-policy assigned — hr-admin group was NOT resolved")
            print("   Check: Is 'groupIds' present in the exchanged token? (see Step 3)")
            print("   Check: Does the group alias name in Vault match the value in groupIds?")
        else:
            print("\n❌ Only 'default' policy assigned — NO group policies resolved")
            print("   groupIds is likely missing from the exchanged token")
        
        VAULT_TOKEN = vault_token
    else:
        print(f"❌ Vault auth failed: {auth_response.text}")
        VAULT_TOKEN = None

🔐 Authenticating with Vault using exchanged token...
HTTP Status: 200

✅ Vault authentication successful

📋 Vault Token Details:
  Token policies    : ['default']
  Identity policies : ['hr-admin-policy']
  All policies      : ['default', 'hr-admin-policy']
  Metadata          : {'role': 'agent-role'}

✅ hr-admin-policy is assigned — Bob should be able to access salary data


---
## 🗄️ Step 5: Test Credential Requests

Try requesting credentials for both `hr-basic-reader` and `hr-admin-reader`.

In [19]:
if not VAULT_TOKEN:
    print("❌ No Vault token available. Complete Step 4 first.")
else:
    # First, look up the token itself to confirm entity_id and effective policies
    print("🔍 Looking up Vault token details...")
    lookup_response = requests.get(
        f"{VAULT_ADDR}/v1/auth/token/lookup-self",
        headers={"X-Vault-Token": VAULT_TOKEN},
        timeout=10
    )
    if lookup_response.status_code == 200:
        token_data = lookup_response.json().get('data', {})
        entity_id = token_data.get('entity_id', '')
        print(f"  entity_id         : {entity_id if entity_id else '❌ MISSING — identity policies will NOT apply!'}")
        print(f"  policies          : {token_data.get('policies', [])}")
        print(f"  identity_policies : {token_data.get('identity_policies', [])}")
        print(f"  display_name      : {token_data.get('display_name', '')}")
        if not entity_id:
            print("\n  ⚠️  No entity_id — Vault did not create an identity entity for this token.")
            print("     Identity group policies (hr-admin-policy) will NOT be enforced.")
    else:
        print(f"  ⚠️  Could not look up token: {lookup_response.text}")

    # Check what capabilities this token actually has on the credential paths
    print("\n🔍 Checking token capabilities on database credential paths...")
    cap_response = requests.post(
        f"{VAULT_ADDR}/v1/sys/capabilities-self",
        headers={"X-Vault-Token": VAULT_TOKEN},
        json={"paths": ["database/creds/hr-basic-reader", "database/creds/hr-admin-reader"]},
        timeout=10
    )
    if cap_response.status_code == 200:
        caps = cap_response.json()
        print(f"  database/creds/hr-basic-reader  : {caps.get('database/creds/hr-basic-reader', ['none'])}")
        print(f"  database/creds/hr-admin-reader  : {caps.get('database/creds/hr-admin-reader', ['none'])}")
    else:
        print(f"  ⚠️  Could not check capabilities: {cap_response.text}")

    print()

    for role in ['hr-basic-reader', 'hr-admin-reader']:
        print(f"\n{'='*60}")
        print(f"🗄️  Requesting credentials for: {role}")
        print('='*60)
        
        creds_response = requests.get(
            f"{VAULT_ADDR}/v1/database/creds/{role}",
            headers={"X-Vault-Token": VAULT_TOKEN},
            timeout=10
        )
        
        print(f"HTTP Status: {creds_response.status_code}")
        
        if creds_response.status_code == 200:
            creds = creds_response.json().get('data', {})
            print(f"✅ Credentials issued:")
            print(f"   Username : {creds.get('username')}")
            print(f"   Password : {'*' * 12} (hidden)")
        else:
            print(f"❌ Denied: {creds_response.json().get('errors', creds_response.text)}")

🔍 Looking up Vault token details...
  entity_id         : 4ee424de-1e35-153e-0cff-ee717b5d9919
  policies          : ['default']
  identity_policies : ['hr-admin-policy']
  display_name      : jwt-646008FWXQ

🔍 Checking token capabilities on database credential paths...
  database/creds/hr-basic-reader  : ['deny']
  database/creds/hr-admin-reader  : ['read']


🗄️  Requesting credentials for: hr-basic-reader
HTTP Status: 403
❌ Denied: ['1 error occurred:\n\t* permission denied\n\n']

🗄️  Requesting credentials for: hr-admin-reader
HTTP Status: 200
✅ Credentials issued:
   Username : v-jwt-6460-hr-admin-qgIf97sTsSbXPkQSXxeM-1772441287
   Password : ************ (hidden)


---
## 📊 Summary

After running all steps, use this table to diagnose the issue:

| Check | Expected | What to fix if wrong |
|---|---|---|
| User token has `groupIds` | `["hr-admin"]` for Bob | Add `groupIds` mapping in Frontend Client → Introspect |
| Exchanged token has `groupIds` | `["hr-admin"]` for Bob | Add `groupIds` mapping in Exchange Client → Introspect |
| Vault auth returns `hr-admin-policy` | Present in `identity_policies` | Check group alias name matches `groupIds` value exactly |
| `hr-admin-reader` capability | `['read']` | Re-run `terraform apply` — policy content in Vault is stale |
| `hr-admin-reader` creds request | `200 OK` | All of the above must pass first |

---

### Root Cause A: Exchanged token missing `groupIds`

If Step 3 shows `groupIds` is missing from the exchanged token:

1. Go to **IBM Verify Admin Console**
2. Navigate to **Applications → Exchange Client** (the Tool/Exchange client)
3. Go to **Sign-on → Endpoint configuration → Introspect**
4. Add attribute mapping:
   - **Attribute name**: `groupIds`
   - **Source**: Attribute mapping
   - **Mapping**: `groupIds` → `groupIds`
5. Save and re-test

---

### Root Cause B: Policy content in Vault is stale

**Symptom:** `hr-admin-policy` appears in `identity_policies` (Step 4) but `capabilities-self` returns `['deny']` for `database/creds/hr-admin-reader` (Step 5).

**Cause:** The policy was uploaded to Vault with old content. The `hr-admin-policy.hcl` file has since been updated, but `terraform apply` has not been re-run.

**Fix:** Re-run Terraform to push the current policy files to Vault:

```bash
cd infrastructure/vault/terraform
terraform apply
```

After applying, re-run Steps 4 and 5 with a fresh Vault token (re-run Step 3 → Step 4 → Step 5).

---

### Root Cause C: hr-admin-group has conflicting policies

**Symptom:** Bob gets both `hr-basic-policy` and `hr-admin-policy` in `identity_policies`.

**Note:** In Vault, policies are **additive** — having both is not a conflict. However, the `hr-admin-group` in Terraform was previously configured with both policies unnecessarily. This has been fixed: `hr-admin-group` now only has `hr-admin-policy`, since `hr-admin-policy` already grants access to both basic and salary data.

Re-run `terraform apply` to apply this change.

---